In [ ]:
#  Scenario: Package Delivery in a City
# Imagine you run a delivery company in a huge city with 100,000 houses. Each house has a
#  unique “address fingerprint” (like the 384‑dimensional vectors in your code).
# Now, when a customer places an order, you need to quickly find the nearest warehouse or
#  delivery hub to their house. Instead of checking every single house one by one (which would 
# be very slow), you use smart strategies:

# 🗺️ HNSW (Hierarchical Navigable Small World)
# - Think of it like a layered city map:
# - Top layer = highways (coarse overview).
# - Middle layer = main roads.
# - Bottom layer = all streets and houses.
# - To find the nearest warehouse, you start at the highway level, zoom into the right district, 
# then finally navigate down to the exact street.
# - This is what IndexHNSWFlat does: it builds a graph of connections so you can “hop” quickly 
# to the right neighborhood.

# 🏘️ IVF (Inverted File Index)
# - Imagine dividing the city into 256 neighborhoods (Voronoi cells).
# - First, figure out which neighborhood the customer’s house belongs to.
# - Then, only search inside that neighborhood instead of the whole city.
# - This is what IndexIVFFlat does: it clusters vectors and searches only in the most relevant 
# clusters.
# - nprobe = 10 means you don’t just check one neighborhood, but the 10 most likely ones —
#  balancing speed and accuracy.

# ⚡ In Plain Words
# - HNSW = zooming in layer by layer on a city map.
# - IVF = splitting the city into neighborhoods and searching locally.
# - Both methods help you find the closest match fast without scanning all 100,000 houses

In [2]:
import faiss
import numpy as np

In [7]:
d = 384
n = 100_000

In [8]:
index = faiss.IndexHNSWFlat(d, 32)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 100

In [9]:
vectors = np.random.randn(n, d).astype("float32")
faiss.normalize_L2(vectors)
index.add(vectors)

In [10]:
query = np.random.randn(1, d).astype("float32")
faiss.normalize_L2(query)
D, I = index.search(query, k=10)    # top-10
print("Indices:", I[0])             # [2341, 8712, ...]
print("Distances:", D[0])  

Indices: [12987 21161 26476 69030 18484 62879 51694 35641 14320 79830]
Distances: [1.5687829 1.6048303 1.6150582 1.6310744 1.6321585 1.633153  1.6331801
 1.6349602 1.6464114 1.6475849]


In [11]:
nlist = 256   # number of Voronoi cells
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)
index_ivf.train(vectors)
index_ivf.add(vectors)
index_ivf.nprobe = 10               # cells to search

In [13]:
# Scenario: Student Course Recommendations
# Imagine you’re running an online learning platform. Students type queries like “beginner-friendly Python tutorials”, 
# but your catalog has courses titled “Introduction to Programming with Python”.
# Traditional keyword search might miss the match, but embeddings + vector search (Chroma locally or Pinecone in the cloud)
#  can connect them semantically.

# 🧩 Teaching Exercise Flow
# - Documents (Courses)
# - "Python is a high-level programming language"
# - "Machine learning models need training data"
# - "Dogs are loyal and friendly animals"
# - "Cats are independent and curious pets"
# - Embedding Model
# - Converts each course description into a vector (numbers capturing meaning).
# - Indexing
# - Chroma (local): Great for classroom demos, no API key needed.
# - Pinecone (cloud): Scales to billions of items, perfect for production.
# - Query
# - Student asks: “What animals make good companions?”
# - Embedding of query is compared against course embeddings.
# - Results
# - Top matches:
# - Cats are independent and curious pets
# - Dogs are loyal and friendly animals

In [14]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()  
collection = client.create_collection("docs")

documents = [
    "Python is a high-level programming language",
    "Machine learning models need training data",
    "Dogs are loyal and friendly animals",
    "Cats are independent and curious pets",
]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# Encode documents into embeddings
embeddings = model.encode(documents).tolist()

# Add to Chroma collection
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc{i}" for i in range(len(documents))],
    metadatas=[{"source": "demo", "idx": i} for i in range(len(documents))]
)

# Query with semantic search
query = "What animals make good companions?"
q_emb = model.encode([query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

# Print results
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[{dist:.3f}] {doc}")

[0.806] Dogs are loyal and friendly animals
[1.224] Cats are independent and curious pets
[1.782] Python is a high-level programming language


In [21]:
# Query with semantic search
query = "Course on training machine learning models"
q_emb = model.encode([query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

# Print results
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[{dist:.3f}] {doc}")

[0.674] Machine learning models need training data
[1.658] Python is a high-level programming language
[1.798] Dogs are loyal and friendly animals


In [22]:
# Query with semantic search
query = "Best animals to keep as pets"
q_emb = model.encode([query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

# Print results
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[{dist:.3f}] {doc}")

[0.892] Dogs are loyal and friendly animals
[1.059] Cats are independent and curious pets
[1.838] Python is a high-level programming language


In [23]:
query = "Animals that are good for families"
q_emb = model.encode([query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

# Print results
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"[{dist:.3f}] {doc}")

[1.136] Dogs are loyal and friendly animals
[1.288] Cats are independent and curious pets
[1.744] Python is a high-level programming language
